<a href="https://colab.research.google.com/github/Shobhan-Kumar-P/LLMs/blob/main/RAG_ChatBot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🤖 Retrieval-Augmented Generation (RAG) Chatbot

This project implements a **Retrieval-Augmented Generation (RAG)** pipeline that enhances Large Language Model (LLM) responses using external knowledge sources.

---

## 🚀 Overview

The system takes a user query, retrieves the most relevant information from a document corpus, and generates a grounded response using an LLM.

Unlike standalone LLMs, this approach:

* Reduces hallucinations
* Improves factual accuracy
* Enables domain-specific question answering

---

## 🧩 Architecture

The pipeline consists of the following stages:

1. **Document Ingestion**

   * Load and preprocess raw text data
   * Split into semantically meaningful chunks (sentence-based)

2. **Embedding Generation**

   * Convert text chunks into vector representations

3. **Vector Search (Retrieval)**

   * Used FAISS for efficient similarity search over embeddings
   * Implemented dense vector retrieval using cosine similarity (via normalized embeddings)
   * Currently using a flat index (IndexFlatIP) for exact nearest neighbor search
   * Retrieves top-k most relevant chunks based on semantic similarity

4. **Context Construction**

   * Combine retrieved chunks into a structured prompt
   * Limit context size to improve response speed

5. **LLM Response Generation**

   * Generate answers grounded strictly in retrieved context

---

## ⚙️ Tech Stack

* **Language**: Python
* **Frameworks**: LangChain
* **Embeddings**: HuggingFace
* **Vector Store**: FAISS
* **LLM**: HuggingFace models

---

## 📊 Features

### ✅ Implemented

* Basic RAG pipeline
* Semantic search using vector embeddings
* Context-aware answer generation

### ⚠️ Current Limitations

* No caching (recomputes results each query)
* Limited optimization for latency
* No evaluation pipeline
* No streaming responses
* Not deployed (runs in Colab environment)

---

## 🚧 Planned Improvements (Production Roadmap)

* [ ] Sentence-aware chunking with overlap
* [ ] Faster ANN indexing (FAISS tuning: IVF/HNSW)
* [ ] Query + response caching (Redis)
* [ ] Hybrid search (vector + keyword)
* [ ] Re-ranking retrieved results
* [ ] Streaming responses for better UX
* [ ] Evaluation framework (RAGAS / custom metrics)
* [ ] API deployment (FastAPI)
* [ ] Docker containerization

---

## ▶️ How to Run

1. Open the notebook in Google Colab
2. Upload your dataset file
3. Run all cells sequentially
4. Enter your query when prompted
5. View the generated response

---

## 📈 Performance Notes

* Current latency: ~3–5 minutes (prototype setup)
* Target latency (production): ~1–3 seconds
* Bottlenecks: model loading, retrieval inefficiency, lack of caching

---

## 🧠 Key Learnings

* Proper chunking significantly impacts retrieval quality
* ANN-based vector search is critical for scalability
* Smaller, optimized models outperform large models in real-time systems
* System design matters more than raw compute (e.g., TPUs)

---

## 📌 Conclusion

This project demonstrates the core principles of RAG systems and serves as a foundation for building scalable, production-ready AI applications. Further optimizations in retrieval, caching, and deployment are required to meet real-world performance standards.


In [ ]:
pip install pymupdf langchain langchain-text-splitters transformers sentence_transformers faiss-cpu torch

In [ ]:
import pymupdf
def extract_text_from_pdf(filepath):
    text = []
    try:
        data = pymupdf.open(filepath)
        for page_num, temp in enumerate(data):
            pdf_text = temp.get_text()
            if pdf_text:
                text.append({"page" : page_num,
                             "content" : pdf_text})
        data.close()
        return text

    except Exception as e:
        print("file path error", e)
        return []

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

def structural_chunking(pages):
    all_chunks = []

    separators = [
        "\n\n",      # paragraph
        "\n",        # line
        ". ",        # sentence end
        "? ",        # question
        "! ",        # exclamation
        " "          # fallback (word-level, safer than "")
    ]

    splitter = RecursiveCharacterTextSplitter(
        chunk_size=400,        # increased for better context
        chunk_overlap=80,      # more overlap = better continuity
        separators=separators
    )

    for page in pages:
        splits = splitter.split_text(page["content"])

        for chunk in splits:
            # optional cleaning
            chunk = chunk.strip()

            if len(chunk) < 50:   # skip very small/noisy chunks
                continue

            all_chunks.append({
                "page": page["page"],
                "content": chunk
            })

    return all_chunks

In [ ]:
from sentence_transformers import SentenceTransformer

def embeddings(chunks):
    content = [chunk["content"] for chunk in chunks]

    model = SentenceTransformer("all-MiniLM-L6-v2")

    embeds = model.encode(
        content,
        batch_size=32,
        normalize_embeddings=True,
        show_progress_bar=True
    )

    all_embeds = []

    for chunk, embed in zip(chunks, embeds):
        all_embeds.append({
            "page": chunk["page"],
            "content": chunk["content"],
            "embedding": embed
        })

    return all_embeds

In [ ]:
import faiss
import numpy as np
import hashlib

def generate_id(text):
    return int(hashlib.md5(text.encode("utf-8")).hexdigest(), 16)%(10**18)


def store_in_faiss_with_ids(embeddings):
    vectors = [embed["embedding"] for embed in embeddings]
    vectors = np.array(vectors).astype("float32")

    dimension = vectors.shape[1]

    faiss.normalize_L2(vectors)

    base_index = faiss.IndexFlatIP(dimension)

    index = faiss.IndexIDMap(base_index)

    ids = np.array([generate_id(id["content"]) for id in embeddings]).astype("int64")

    index.add_with_ids(vectors, ids)

    metadata_store = {
        int(id) : {
            "page" : chunk["page"],
            "content" : chunk["content"]
        } for id, chunk in zip(ids, embeddings)
    }

    return index, metadata_store

In [ ]:
def build_prompt(query, retrieved_chunks):
    context = ""
    for temp in retrieved_chunks:
        context = context + temp["content"] + "\n\n"

    prompt = f"""
        Answer ONLY from the given context.
        If answer is not in context, say "I don't know".

        Context:
        ----------------
        {context}
        ----------------

        Question: {query}

        Answer:
        """
    return prompt

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, AutoConfig
import torch

def load_llm():
    model_name = "microsoft/phi-2"

    # 🔹 Load tokenizer
    tokenizer = AutoTokenizer.from_pretrained(
        model_name,
        trust_remote_code=True
    )

    # ✅ Set padding token
    tokenizer.pad_token = tokenizer.eos_token

    # 🔹 Load config FIRST and fix it
    config = AutoConfig.from_pretrained(
        model_name,
        trust_remote_code=True
    )
    config.pad_token_id = config.eos_token_id

    # 🔹 Load model with fixed config
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        config=config,
        torch_dtype=torch.float32,  # for CPU
        trust_remote_code=True
    )

    # ✅ Also set generation config (extra safety)
    model.generation_config.pad_token_id = tokenizer.eos_token_id

    model.eval()

    return tokenizer, model

In [ ]:
def generate_response(prompt, tokenizer, model):
    inputs = tokenizer(prompt, return_tensors="pt")

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=50,
            temperature=0.3,
            pad_token_id=tokenizer.eos_token_id  # extra safety
        )

    # ✅ Better decoding (no fragile slicing)
    generated_tokens = outputs[0][inputs["input_ids"].shape[1]:]
    response = tokenizer.decode(generated_tokens, skip_special_tokens=True)

    return response.strip()


In [ ]:
def retrieve(query, model, index, metadata_store, top_k = 3):
    query_vector = model.encode([query],
                                normalize_embeddings = True).astype("float32")

    scores, idx = index.search(query_vector, top_k)
    results = []
    for ids in idx[0]:
        if ids == -1:
            continue

        results.append(metadata_store[int(ids)])

    return results

In [ ]:
#import sys

from sentence_transformers import SentenceTransformer

def main():

    model = SentenceTransformer("all-MiniLM-L6-v2")

    data = extract_text_from_pdf(r"/content/frvt_1N_report.pdf")

    chunks = structural_chunking(data)


    embeds = embeddings(chunks)

    index, metadata_store = store_in_faiss_with_ids(embeds)

    while True:
        query = input("Ask : ")
        if query.lower() in ["exit", "quit"]:
          break
        #query = "what are false positives?"

        retrieve_chunks = retrieve(query, model, index, metadata_store, 5)

        for i in retrieve_chunks:
          print(i)
          print("___________________________________________")

        prompt = build_prompt(query, retrieve_chunks)

        print("______%%%%%%_______************")
        print(prompt)

        #sys.exit()

        tokenizer, llm = load_llm()

        response = generate_response(prompt, tokenizer, llm)

        print("response : ", response)

if __name__ == "__main__":
    main()